# OpenAI Agents SDK — Agent, Runner, tools, sessions, handoffs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USER/SA_LIB_WORKSHOP_Day2/blob/main/03_openai_agents_sdk/03_openai_agents_sdk.ipynb)

**SA-LIB AI Workshop 2026 — Day 2: AI Agents**

OpenAI's lightweight agent framework. Works with **any** OpenAI-compatible provider through `OpenAIChatCompletionsModel`, so Groq, Gemini, DeepSeek and Ollama all work.

Same example in every framework: **(1) first agent → (2) tools → (3) memory → (4) fallback → (5) multi-agent → complete agent**.

## 0. Setup
Install the packages (takes ~1 minute in Colab).

In [ ]:
%pip install -q "openai-agents>=0.2" "python-dotenv" "ddgs"

### API keys
* **Colab:** click the 🔑 *Secrets* icon on the left, add e.g. `GROQ_API_KEY`, and enable notebook access.
* **VS Code / Jupyter / Linux:** put your keys in the `.env` file in the repo root (copy `.env.example`).
* Otherwise you'll be asked to paste a key below.

Choose the model provider with `LLM_PROVIDER`: `groq` · `gemini` · `openai` · `deepseek` · `openrouter` · `ollama`.

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # VS Code / Linux: read ../.env
try:                                           # Colab: read the Secrets panel
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'DEEPSEEK_API_KEY', 'OPENROUTER_API_KEY', 'LANGSMITH_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST', 'LLM_PROVIDER', 'LLM_MODEL']:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("LLM_PROVIDER", "groq")   # <- change the provider here if you like
provider = os.environ["LLM_PROVIDER"]
key_name = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY",
            "deepseek": "DEEPSEEK_API_KEY", "openrouter": "OPENROUTER_API_KEY"}.get(provider)
if key_name and not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"{key_name}: ")
print("provider:", provider)

### Shared helper: `workshop_common.py`
Reads the provider settings, and holds the two tools every example uses: `web_search` (free DuckDuckGo) and `word_count`.

In [ ]:
%%writefile workshop_common.py
"""
workshop_common.py — shared helpers for every framework folder in this workshop.

(the SAME file is copied into each folder so every folder works on its own —
 edit shared/workshop_common.py and run `python tools/sync_common.py` to update the copies)

What it gives you:
  get_llm_config()   -> which provider / model / base_url / api key to use (read from .env)
  fallback_order()   -> the list of providers to try, in order, for the fallback examples
  ping()             -> quick "are you alive?" check against a provider
  web_search()       -> free DuckDuckGo search (no key)
  word_count()       -> the tiny "tool you wrote yourself" from the original guide

Every provider below speaks the OpenAI chat-completions protocol, which is why every
framework in this workshop can use every provider: we just change base_url + api_key + model.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

# walks up from where you run python, so the .env in the repo root is always found
load_dotenv(find_dotenv(usecwd=True))

PROVIDERS = {
    #  name        base_url                                                     api-key env var       default model
    "groq":       ("https://api.groq.com/openai/v1",                           "GROQ_API_KEY",       "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY",     "gemini-2.5-flash"),
    "openai":     ("https://api.openai.com/v1",                                "OPENAI_API_KEY",     "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com",                                 "DEEPSEEK_API_KEY",   "deepseek-chat"),
    "openrouter": ("https://openrouter.ai/api/v1",                             "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),  None,                 "qwen2.5:7b"),
}


@dataclass
class LLMConfig:
    provider: str
    model: str
    base_url: str
    api_key: str

    @property
    def litellm_model(self) -> str:
        """Model name in LiteLLM format (used by CrewAI and Google ADK): 'openai/<model>'
        means 'an OpenAI-compatible endpoint' — combined with base_url it works for every provider."""
        return f"openai/{self.model}"


def get_llm_config(provider: str | None = None) -> LLMConfig:
    """Read LLM_PROVIDER / LLM_MODEL / <PROVIDER>_API_KEY from the environment."""
    provider = (provider or os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown LLM_PROVIDER '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    base_url, key_env, default_model = PROVIDERS[provider]
    api_key = os.getenv(key_env) if key_env else "ollama"  # ollama runs locally, no key
    if not api_key:
        raise RuntimeError(f"{key_env} is missing — add it to your .env file (see .env.example).")
    # LLM_MODEL only overrides the model of the MAIN provider, not of the fallbacks
    main_provider = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    model = (os.getenv("LLM_MODEL") if provider == main_provider else None) or default_model
    return LLMConfig(provider, model, base_url, api_key)


def fallback_order() -> list[str]:
    """Providers to try, in order. Default: your LLM_PROVIDER first, then LLM_FALLBACKS (groq,gemini)."""
    main = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    extra = [p.strip().lower() for p in os.getenv("LLM_FALLBACKS", "groq,gemini").split(",") if p.strip()]
    return list(dict.fromkeys([main, *extra]))  # remove duplicates, keep order


def ping(cfg: LLMConfig) -> None:
    """Send a 1-token request. Raises an exception if the provider is down, rate-limited or the key is wrong."""
    from openai import OpenAI

    OpenAI(base_url=cfg.base_url, api_key=cfg.api_key, timeout=20, max_retries=0).chat.completions.create(
        model=cfg.model, messages=[{"role": "user", "content": "ping"}], max_tokens=1
    )


def web_search(query: str) -> str:
    """Search the web with DuckDuckGo (free, no key) and return the top results as text."""
    try:
        from ddgs import DDGS

        results = DDGS().text(query, max_results=3)
        return "\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # the free search is rate-limited: return the error instead of crashing
        return f"web search failed ({e}). try again in a few seconds."


def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

## 1. `01_first_agent.py`

OpenAI Agents SDK 01 — your first agent
the SDK has 3 ideas: Agent (instructions + tools), Runner (runs the loop), and results.
it is built for OpenAI, but OpenAIChatCompletionsModel lets it use ANY OpenAI-compatible provider.

In [ ]:
import asyncio
import os

from agents import Agent, AsyncOpenAI, OpenAIChatCompletionsModel, Runner, set_tracing_disabled

from workshop_common import get_llm_config

# tracing uploads runs to platform.openai.com — only possible with an OpenAI key
set_tracing_disabled(not os.getenv("OPENAI_API_KEY"))

cfg = get_llm_config()
model = OpenAIChatCompletionsModel(model=cfg.model, openai_client=AsyncOpenAI(base_url=cfg.base_url, api_key=cfg.api_key))

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Be concise and accurate.",
    model=model,
)


async def main():
    result = await Runner.run(agent, "explain what an AI agent is in two sentences.")
    print(f"[{cfg.provider}:{cfg.model}]", result.final_output)


if __name__ == "__main__":
    await main()

## 2. `02_agent_with_tools.py`

OpenAI Agents SDK 02 — giving your agent tools
@function_tool turns any python function into a tool (name, docstring and type hints -> JSON schema).

In [ ]:
import asyncio
import os

from agents import Agent, AsyncOpenAI, OpenAIChatCompletionsModel, Runner, function_tool, set_tracing_disabled

import workshop_common as wc

set_tracing_disabled(not os.getenv("OPENAI_API_KEY"))
cfg = wc.get_llm_config()
model = OpenAIChatCompletionsModel(model=cfg.model, openai_client=AsyncOpenAI(base_url=cfg.base_url, api_key=cfg.api_key))


@function_tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


@function_tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Use your tools when they help answer accurately.",
    tools=[web_search, word_count],
    model=model,
)


async def main():
    result = await Runner.run(agent, "search for the latest langchain version, then tell me how many words your answer is.")
    for item in result.new_items:          # every step of the loop: tool calls, tool outputs, messages
        print(f"[{item.type}]")
    print(result.final_output)


if __name__ == "__main__":
    await main()

## 3. `03_agent_with_memory.py`

OpenAI Agents SDK 03 — giving your agent memory
a Session stores the conversation. SQLiteSession("chat-1") = the thread_id idea from LangGraph.
pass a file path as 2nd argument (e.g. "memory.db") to keep memory after the script stops.

In [ ]:
import asyncio
import os

from agents import Agent, AsyncOpenAI, OpenAIChatCompletionsModel, Runner, SQLiteSession, set_tracing_disabled

from workshop_common import get_llm_config

set_tracing_disabled(not os.getenv("OPENAI_API_KEY"))
cfg = get_llm_config()
model = OpenAIChatCompletionsModel(model=cfg.model, openai_client=AsyncOpenAI(base_url=cfg.base_url, api_key=cfg.api_key))

agent = Agent(name="Assistant", instructions="You are a helpful assistant. Be concise and accurate.", model=model)


async def main():
    session = SQLiteSession("chat-1")      # in-memory SQLite; same session = same memory
    r1 = await Runner.run(agent, "hi! my name is m0h.", session=session)
    print(r1.final_output)
    r2 = await Runner.run(agent, "what's my name?", session=session)
    print(r2.final_output)


if __name__ == "__main__":
    await main()

## 4. `04_agent_with_fallback.py`

OpenAI Agents SDK 04 — the fallback setup

In [ ]:
import asyncio
import os

from agents import Agent, AsyncOpenAI, OpenAIChatCompletionsModel, Runner, set_tracing_disabled

from workshop_common import fallback_order, get_llm_config, ping

set_tracing_disabled(not os.getenv("OPENAI_API_KEY"))


def get_model():
    for provider in fallback_order():
        try:
            cfg = get_llm_config(provider)
            ping(cfg)
            print(f"using {provider} ({cfg.model})")
            client = AsyncOpenAI(base_url=cfg.base_url, api_key=cfg.api_key)
            return OpenAIChatCompletionsModel(model=cfg.model, openai_client=client)
        except Exception as e:
            print(f"{provider} failed ({str(e)[:80]}); trying the next one")
    raise RuntimeError("every provider failed — check your keys in .env")


agent = Agent(name="Assistant", instructions="You are a helpful assistant. Be concise.", model=get_model())


async def main():
    result = await Runner.run(agent, "say hi and tell me which model you are.")
    print(result.final_output)


if __name__ == "__main__":
    await main()

## 5. `05_multi_agent.py`

OpenAI Agents SDK 05 — multi-agent, two patterns

 A) AGENTS AS TOOLS (manager pattern): a manager agent calls specialist agents like tools,
    gets their answers back, and stays in control.
 B) HANDOFFS: a triage agent hands the WHOLE conversation over to a specialist,
    who then answers the user directly.

In [ ]:
import asyncio
import os

from agents import Agent, AsyncOpenAI, OpenAIChatCompletionsModel, Runner, function_tool, set_tracing_disabled

import workshop_common as wc

set_tracing_disabled(not os.getenv("OPENAI_API_KEY"))
cfg = wc.get_llm_config()
model = OpenAIChatCompletionsModel(model=cfg.model, openai_client=AsyncOpenAI(base_url=cfg.base_url, api_key=cfg.api_key))


@function_tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


@function_tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


researcher = Agent(name="Researcher", model=model, tools=[web_search],
                   instructions="Find current facts with web_search. Reply with 3 short bullet facts.")
writer = Agent(name="Writer", model=model, tools=[word_count],
               instructions="Write one friendly paragraph under 60 words. Check the length with word_count.")

# ---- A) agents as tools ----
manager = Agent(
    name="Manager",
    model=model,
    instructions="First ask the researcher for facts, then give those facts to the writer. Return the writer's paragraph.",
    tools=[
        researcher.as_tool(tool_name="research", tool_description="Research a topic on the web."),
        writer.as_tool(tool_name="write", tool_description="Write a short paragraph from facts."),
    ],
)

# ---- B) handoffs ----
math_tutor = Agent(name="Math tutor", model=model, handoff_description="Questions about maths",
                   instructions="You explain maths step by step, briefly.")
history_tutor = Agent(name="History tutor", model=model, handoff_description="Questions about history",
                      instructions="You explain history clearly and briefly.")
triage = Agent(name="Triage", model=model, handoffs=[math_tutor, history_tutor],
               instructions="Decide which tutor should answer and hand off to them. Do not answer yourself.")


async def main():
    print("=== A) agents as tools ===")
    result = await Runner.run(manager, "Write about the latest langchain version.")
    print(result.final_output)

    print("\n=== B) handoffs ===")
    result = await Runner.run(triage, "Who founded the ancient city of Cyrene in Libya?")
    print(f"answered by: {result.last_agent.name}\n{result.final_output}")


if __name__ == "__main__":
    await main()

## 6. `agent.py`

OpenAI Agents SDK — the complete agent (fallback + tools + memory)

In [ ]:
import asyncio
import os

from agents import (Agent, AsyncOpenAI, OpenAIChatCompletionsModel, Runner, SQLiteSession, function_tool,
                    set_tracing_disabled)

import workshop_common as wc

set_tracing_disabled(not os.getenv("OPENAI_API_KEY"))


def get_model():
    for provider in wc.fallback_order():
        try:
            cfg = wc.get_llm_config(provider)
            wc.ping(cfg)
            print(f"using {provider} ({cfg.model})")
            client = AsyncOpenAI(base_url=cfg.base_url, api_key=cfg.api_key)
            return OpenAIChatCompletionsModel(model=cfg.model, openai_client=client)
        except Exception as e:
            print(f"{provider} failed ({str(e)[:80]}); trying the next one")
    raise RuntimeError("every provider failed — check your keys in .env")


@function_tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


@function_tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Use your tools when they help answer accurately, and remember the conversation.",
    tools=[web_search, word_count],
    model=get_model(),
)


async def main():
    session = SQLiteSession("chat-1")
    r1 = await Runner.run(agent, "my name is m0h. search for the latest langchain version, then tell me how many "
                                 "words your answer is.", session=session)
    print(r1.final_output)
    r2 = await Runner.run(agent, "what's my name?", session=session)
    print(r2.final_output)


if __name__ == "__main__":
    await main()

---
*SA-LIB AI Workshop 2026 · based on [build-ai-agents-free](https://github.com/Moh4696/build-ai-agents-free) by m0h (MIT)*